# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [5]:
# Code here
import pandas as pd
import numpy as np
from sklearn import preprocessing as pp
from sklearn.impute import KNNImputer

df = pd.read_csv("./Income Dirty Data.csv")
df.head()

,ID,sex,age,income,tax_15_pct
0,1,Women,21,147168.0,22075.20
1,2,Female,29,119595.0,17939.25
2,3,Female,56,87770.0,13165.50
3,4,NaN,21,54259.0,8138.85
4,5,Male,28,NaN,160230.00


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [25]:
# Code here
print(df.isna().sum())
print("\n")
print(df.isna().sum().sum() / df.size*100, "%")
print("\n")
print(df.dropna().shape[0])



ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64


5.800000000000001 %


733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [29]:
# Code here
adults = (df["age"] >= 18)
tax15 = (df["tax_15_pct"] == df["income"] * 0.15)
have_income = (df["income"] > 0)

rules = ( adults & tax15 & have_income)

print("This amount of people follow the rules:", rules.mean() * 100, "%")



This amount of people follow the rules: 47.0 %


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [39]:
# Code here
df["sex"] = df["sex"].replace({
    "Woman": "Female",
    "Women": "Female",
    "Man": "Female",
    "Men": "Female"
})
print("The sex column has been standardized")



The sex column has been standardized


,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [37]:
# Code here
df.loc[df["age"] <= 0, "age"] = np.nan
df.loc[df["income"] <= 0, "income"] = np.nan
df.loc[df["tax_15_pct"] <= 0, "tax_15_pct"] = np.nan
df.head()

,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [51]:
df_knn = df.copy()

df_knn["sex"] = df_knn["sex"].replace({
    "Female": 0,
    "Male": 1
})

scaled = pp.StandardScaler()
df_scaled = scaled.fit_transform(df_knn)

print("Before imputing:")
print(df_knn.head(10))

imputer = KNNImputer(n_neighbors=5)
df_imputed = imputer.fit_transform(df_scaled)
df_imputed = scaled.inverse_transform(df_imputed)
df_imputed = pd.DataFrame(df_imputed, columns=df_knn.columns)

print("After imputing:")
print(df_imputed.head(10))

Before imputing:
   ID  sex   age    income  tax_15_pct
0   1    0  21.0  147168.0    22075.20
1   2    0  29.0  119595.0    17939.25
2   3    0  56.0   87770.0    13165.50
3   4  NaN  21.0   54259.0     8138.85
4   5    1  28.0       NaN   160230.00
5   6    0   NaN  128326.0    19248.90
6   7    0   NaN       NaN         NaN
7   8    0  24.0       NaN    11820.60
8   9    0  38.0  149473.0    22420.95
9  10    0  48.0  113663.0  1136630.00
After imputing:
     ID  sex   age    income  tax_15_pct
0   1.0  0.0  21.0  147168.0    22075.20
1   2.0  0.0  29.0  119595.0    17939.25
2   3.0  0.0  56.0   87770.0    13165.50
3   4.0  0.4  21.0   54259.0     8138.85
4   5.0  1.0  28.0   86970.2   160230.00
5   6.0  0.0  37.2  128326.0    19248.90
6   7.0  0.0  39.8  115176.2   240657.19
7   8.0  0.0  24.0   84891.0    11820.60
8   9.0  0.0  38.0  149473.0    22420.95
9  10.0  0.0  48.0  113663.0  1136630.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

Inaccurate or missing data can cause incorrect results. think if we wanted to pull up how many females were in the database. The result wouldn't be accurate because some are put down as woman or women.

10. Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your name